# 🧠 Lab 2 : SQL Avancé (UNNEST & Window Functions)

**Objectifs :**
- Gérer des données JSON/imbriquées (`STRUCT` et `ARRAY`)
- Comprendre et utiliser `UNNEST`
- Utiliser une Window Function (`ROW_NUMBER()`) pour un cas d'usage E-commerce

*(Rappel : Tous vos travaux s'effectuent dans votre unique dataset `thelook_prenom`. Les couches sont représentées par les préfixes des tables : `bronze_`, `silver_`, `gold_`)*

## Partie 1 : Les Tableaux et UNNEST()
Dans Google Cloud, les données Web (comme GA4 ou Firebase) sont souvent des tableaux d'objets (Arrays of Structs). Créons de fausses données imbriquées :

In [ ]:
/* ⚠️ Remplacez `thelook_prenom` */
CREATE OR REPLACE TABLE `thelook_prenom.bronze_nested_orders` AS
SELECT 
  'ORD999' as order_id, 
  'USER_1' as user_id,
  [STRUCT('T-shirt' as product, 2 as quantity, 20.0 as price),
   STRUCT('Jeans' as product, 1 as quantity, 50.0 as price)] as items
UNION ALL
SELECT 'ORD888', 'USER_2', [STRUCT('Sneakers' as product, 1 as quantity, 80.0 as price)];

Si vous interrogez cette table `SELECT *`, la colonne `items` contient une liste. Pour analyser les ventes par produit, il faut **aplatir** cette liste avec `UNNEST()`.

**Exemple de UNNEST :**
```sql
SELECT order_id, item.product, item.price
FROM `thelook_prenom.bronze_nested_orders`,
UNNEST(items) AS item;
```

### ✍️ À vous de jouer (Exercice 1)
**Consignes :**
Écrivez une requête qui aplatit la table `bronze_nested_orders` et calcule le revenu total généré par CHAQUE produit (revenu = `quantity * price`).
Enregistrez le résultat dans `thelook_prenom.gold_product_revenue`.

In [ ]:
-- Écrivez votre requête d'aplatissement et agrégation ici :



## Partie 2 : Les Window Functions
Les fonctions de fenêtrage permettent de faire des calculs sur un ensemble de lignes (la fenêtre) liées à la ligne actuelle, sans réduire le nombre de lignes comme le ferait un `GROUP BY`.

### 💡 Cas d'usage E-commerce : Trouver la dernière commande d'un client.
Imaginez qu'un client passe plusieurs commandes et que vous vouliez isoler uniquement sa **commande la plus récente**. 

**L'Exemple Parfait : `ROW_NUMBER()`**
```sql
WITH RankedOrders AS (
  SELECT 
    order_id,
    user_id,
    order_date,
    ROW_NUMBER() OVER(PARTITION BY user_id ORDER BY order_date DESC) as rank_commande
  FROM `thelook_prenom.silver_clean_orders_items`
)
SELECT *
FROM RankedOrders
WHERE rank_commande = 1; -- 1 = La plus récente pour chaque utilisateur !
```

### ✍️ À vous de jouer (Exercice 2)
**Consignes :**
En utilisant les données de `thelook_prenom.silver_clean_orders_items` (créée au Lab 1), écrivez une requête avec une Window Function pour calculer le rang de chaque article par utilisateur, mais cette fois-ci **basé sur le montant le plus élevé** (`sale_price` du plus cher au moins cher). 

Filtrez ensuite pour n'afficher que l'article le plus cher jamais acheté par chaque client.

In [ ]:
-- Écrivez votre requête avec Window Function ici :

